# Adapted Extraction of Spectral Features: Glides & Notes

#### Import the packages.

In [ ]:
# base packages
from itertools import groupby
from pathlib import Path
from glob import glob
from time import sleep
# standard pip packages
import matplotlib.pyplot as plt
import numpy as np
import scipy
import pandas as pd
# special packages
import librosa
import librosa.display
import mir_eval
import soundfile

#### Load the songs as wav audio files.

In [ ]:
# Specify directory and local paths to all improv songs of a participant.
subject = ''
condition = '' ## UNCOMMENT THE CORRECT SONG-ID code part at the end.

# directory = Path(f"C:/Users/monik/Documents/Paris/Internship/Data-Collection/Earworms/Harmonic/{subject}/{condition}/")
directory = Path(f"C:/Users/monik/Documents/Paris/Internship/Data-Collection/Earworms/Harmonic2")
filepaths = directory.glob("*.wav")
filepaths = list(filepaths)                                              
filenames = [fp.name for fp in filepaths]

# Sort the songs in order from 1st to 30th
filenames.sort()

# Read in the audio files
datapath = directory.as_posix()

In [ ]:
# Quick check for order of songs.
print(datapath)
print(len(filenames))
print(filenames)

## Extract relevant spectral features.

#### Load audio files to librosa.

In [ ]:
# Load the audio files to librosa for analysis.
y_all = []
sr_all = [] # Fixed to 22050 for all songs.

for filepath in filepaths:
    y, sr = librosa.load(filepath)
    y_all.append(y)
    sr_all.append(sr)

### Use last file for plotting. ###
# Compute the spectrogram.
D = librosa.stft(y)

# Convert an amplitude spectrogram to dB-scaled SPECTROGRAM.
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max) 

In [ ]:
# Check the sampling rate and the length of y_all. There should be 30 y-s for IMPROV and 16 for SONG-FORM 
print(len(y_all))
print(np.unique(sr_all))

## Extract timeframe, fundamental frequency, and amplitude per song.

In [ ]:
# Initialize empty list for lopping through all songs.
sr = sr_all[0] # Constant for all songs.
tf_all = [] # Time frame for all songs per participant.
f0_all = [] # Fundamental frequency for all songs per participant.
midi_pitch_all = []
ng_samples_all = []  # Digitized f0 stabilizing notes and transitions
ng_type_all = []  # Classification labels of each unit
ng_id_all = []  # id number for each note and glide


# Set a progress bar that will tell you at which song the program is and how much is left.
length_bar = len(y_all)
bar = '-'*length_bar
replace_no = len(y_all)/length_bar

# Set the frame size and hop size for pitch analysis
frame_size = 5
hop_size = 1  


for i in range(len(y_all)): # Loop through all songs of this participant to get f0 of each song.
    
    # Print a progress bar.
    num_squares = int(i/replace_no)
    bar = '#'*num_squares+'-'*(length_bar-num_squares)
    percentage = round((i*100)/(len(y_all)-1), 2) # round to 2 int
    sleep(.1)
    print(bar, percentage, '%', end='\r')
    
    #### Get f0. ####
    f0, vid, vpd = librosa.pyin(
        y_all[i],
        sr=sr,
        fmin=librosa.note_to_hz("C1"), # The min and max are restrictive - lower & higher values are feasible.
                                       # The recommended min value = C1, recommended max value = C7.
        fmax=librosa.note_to_hz("C7"),
        frame_length=2048,
        resolution=0.1, # 0.01 corresponds to cent resolution but it takes longer, change if necessary for other analyses
    )
    # convert to log2 format and remove outliers
    midi_pitch = librosa.core.hz_to_midi(f0)
    rng = np.nanmean(midi_pitch) - 3*np.nanstd(midi_pitch), np.nanmean(midi_pitch) + 3*np.nanstd(midi_pitch)
    midi_pitch[np.logical_or(midi_pitch < rng[0], midi_pitch > rng[1])] = np.nan
    f0[np.logical_or(midi_pitch < rng[0], midi_pitch > rng[1])] = np.nan
    f0_all.append(f0)
    midi_pitch_all.append(midi_pitch)
    
    # Create a numpy array for time frames.
    tf = np.linspace(start=0, stop =(len(y_all[i]) / sr), num=len(f0))
    tf_all.append(tf)
    
    ### Analyze f0 into notes (stable maintenance of f0) and glides (transitions from one note to the next)
    # Compute the mean slope of pitch within each frame
    pitch_slope = []
    for i in range(0, len(midi_pitch) - frame_size, hop_size):
        frame_pitch = midi_pitch[i:i+frame_size]
        frame_slope = np.nanmean(np.diff(frame_pitch)/tf[1])
        pitch_slope.append(frame_slope)

    # Find the stable and gliding periods based on a pitch slope threshold
    pitch_slope_threshold = 12  # Adjust this threshold to control sensitivity
    notes_and_glides = []
    is_stable = False
    is_glide = False
    stable_end = False
    glide_end = False
    ng_counter = 0  # counter to make an ID for each event (note or glide)
    # saving variables
    ng_id = np.full_like(tf, np.nan)  # Event ID
    ng_type = np.full_like(tf, '', dtype=object)  # Event type (note or glide)
    ng_samples = np.full_like(tf, np.nan)  # New frequency representation
    for i, slope in enumerate(pitch_slope):
        if np.isnan(slope) or i == len(pitch_slope) - 1: # if no pitch end whatever was going.
            if is_stable:
                stable_end = True
            if is_glide:
                glide_end = True
        elif abs(slope) <= pitch_slope_threshold: # Stable period
    #         import pdb; pdb.set_trace()
            if not is_stable:  # Start a new stable period
                is_stable = True
                start = i * hop_size
            if is_glide: # if we were gliding, we're not any more. 
                glide_end = True
        else:  # Gliding period
            if not is_stable or i * hop_size - start < (0.1 / tf[1]):  # Ignore isolated gliding frames and short gliding periods
                continue
            else:  # End of a stable period, start of a gliding period
                is_glide = True
                glide_start = i * hop_size
                glide_pitch_start = midi_pitch[i]
                stable_end = True

        if stable_end:
            end = i * hop_size
            notes_and_glides.append((start, end, np.nanmedian(midi_pitch[start:end])))
            is_stable = False
            stable_end = False
            ng_id[start+frame_size//2:end+frame_size//2] = ng_counter
            ng_type[start+frame_size//2:end+frame_size//2] = 'note'
            ng_samples[start+frame_size//2:end+frame_size//2] = np.nanmedian(midi_pitch[start:end])
            ng_counter += 1
            
        if glide_end:
            glide_ending = i * hop_size
            glide_pitch_end = midi_pitch[i]
            notes_and_glides.append((glide_start, glide_ending, glide_pitch_start, glide_pitch_end))
            is_glide = False
            glide_end = False
            ng_id[glide_start+frame_size//2:glide_ending+frame_size//2] = ng_counter
            ng_type[glide_start+frame_size//2:glide_ending+frame_size//2] = 'glide'
            ng_samples[glide_start+frame_size//2:glide_ending+frame_size//2] = np.linspace(glide_pitch_start, glide_pitch_end, glide_ending-glide_start)
            ng_counter += 1
    ng_samples_all.append(ng_samples)
    ng_id_all.append(ng_id)
    ng_type_all.append(ng_type)

### Check the output for validity.

In [ ]:
# plot spectrogram
fig, ax = plt.subplots(figsize=(10, 4))
img = librosa.display.specshow(S_db, x_axis="time", y_axis="log", ax=ax, sr=sr)

# fig.colorbar(img, ax=ax, format="%2.f dB")
tf = np.linspace(0, len(y) / sr, len(f0))
ax.plot(tf, f0)
ax.set(title=filenames[-1], ylim=[64, 1024], xlim=[20, 25])
fig.show()
#fig.savefig(f"Spectrogram-{subject}-{condition}.png", dpi=300)

#### Check Outlier Removal in the f0 estimation.

In [ ]:
fig, ax = plt.subplots()
ptch = librosa.core.hz_to_midi(f0)
rng = np.nanmean(ptch) - 3*np.nanstd(ptch), np.nanmean(ptch) + 3*np.nanstd(ptch)
n, bins, _ = ax.hist(ptch, 50)
n, bins, _ = ax.hist(ptch[np.logical_or(ptch < rng[0], ptch > rng[1])], bins)
[ax.axvline(r) for r in rng]
ptch[np.logical_or(ptch < rng[0], ptch > rng[1])] = np.nan

#### Plot note and glide notation against true f0 to check validity.

In [ ]:
plt.plot(tf, ptch, 'k')
for n in np.unique(ng_id):
    if ~np.isnan(n):
        indx = ng_id == n
        if ng_type[indx][0] == 'note':
            color = 'C0'
        elif ng_type[indx][0] == 'glide':
            color = 'C1'
        plt.plot(tf[indx], ng_samples[indx], color)

In [ ]:
# # Plot note histogram
import seaborn as sns
sns.set(style="whitegrid", palette = 'pastel')

chroma_test = np.mod(np.concatenate(ng_samples_all) - 60, 12)
chroma_test[np.concatenate(ng_type_all) == 'glide'] = np.nan
plt.hist(chroma_test, 50)
plt.xlabel('Chroma from f0 stability analysis')

fig, axs = plt.subplots(nrows=4, ncols=4, sharex=True, figsize=(10, 6), tight_layout=True)
for song, ng_samples in enumerate(ng_samples_all):
    chroma_test = np.mod(ng_samples - 60, 12)
    chroma_test[ng_type_all[song] == 'glide'] = np.nan
    axs.flatten()[song].hist(chroma_test, 20, density=True)
    plt.suptitle(f'Histograms of Notes for {condition} songs of {subject}')
    
for i, ax in enumerate(axs.ravel(), 1):
    ax.set_title("Song #{}".format(i))
    
#fig.savefig(f"Note-Histogram-{subject}-{condition}.png", dpi=300)

## Save extracted data.

In [ ]:
# Initialize empty dataframe for spectral features.
df_spectral = pd.DataFrame()

# Add columns with the extracted spectral features
df_spectral['TIME_FRAME'] = pd.DataFrame(data=np.concatenate(tf_all))
df_spectral['FUNDAMENTAL_FREQUENCY'] = pd.DataFrame(data=np.concatenate(f0_all))
df_spectral['MIDI_NOTE'] = pd.DataFrame(data=np.concatenate(midi_pitch_all))
df_spectral['NG_type'] = np.concatenate(ng_type_all) # glide or note
df_spectral['NG_ID'] = np.concatenate(ng_id_all) # order
df_spectral['NG'] = np.concatenate(ng_samples_all)

# Extract participant code.
participant_ID = []

for filename in filenames:
    split_first = filename.split('_recording_mic_')
    participant_ID.append(split_first[0])

participant_ID_spectral = participant_ID*len(np.concatenate(tf_all))

# Add a column with the ID of the participant.
# df_spectral['PARTICIPANT_ID'] = subject

# Add song number information. ------------------------------------------------------  # Code for Improv conditions
# song_ID = []
# for count, sample in enumerate(tf_all, 1): # Start the counter with 1 instead of 0.
#     len_sample = len(sample)
#     song_no = [count]*len_sample
#     song_ID.append(song_no)

# song_ID = np.concatenate(song_ID)
# df_spectral['SONG_ID'] = pd.DataFrame(data=song_ID)

## Add song title for inspiration songs and earworms -------------------------------- # Code for Earworms
# song_ID = []
# for filename in filenames:
#     s1 = filename.split('_')
#     s2 = s1[-1].split('.')
#     s3 = [str(s2[0])]
#     len_sample = len(sample)
#     song_no = s3*len_sample
#     song_ID.append(song_no)
    
# song_ID = np.concatenate(song_ID)
# df_spectral['SONG_ID'] = pd.DataFrame(data=song_ID)

# Add song number information. ------------------------------------------------------  # Code for Song-form conditions
# song_ID = []
# for count, sample in enumerate(tf_all, 1): # Start the counter with 1 instead of 0.
#     len_sample = len(sample)
#     song_no = [count]*len_sample
#     if count <= 4:
#         for el in list(song_no):
#             song_no = 'Lullaby'+'_'+str(el) # Dance-EN, Berceuse-FR
#             song_ID.append(song_no)
#     elif count <= 8:
#         for el in list(song_no):
#             song_no = 'Love'+'_'+str(el-4) # Healing-EN, Chanson d'amour-FR
#             song_ID.append(song_no)
#     elif count <= 12:
#         for el in list(song_no):
#             song_no = 'Dance'+'_'+str(el-8) # Love-EN, Chanson de danse-FR
#             song_ID.append(song_no)
#     else:
#         for el in list(song_no):
#             song_no = 'Healing'+'_'+str(el-12) # Lullaby-EN, Chanson de guérison-FR
#             song_ID.append(song_no)
            
# df_spectral['SONG_TYPE'] = pd.DataFrame(data=song_ID)

# Tests
df_spectral.info()
df_spectral.head()

In [ ]:
df_spectral['SONG_ID'].unique()

In [ ]:
# Save all dataframes to csv.
# df_spectral.to_csv(f"note-glide_data_{subject}_{condition}.csv") # IMPROV or SONG-FORM
df_spectral.to_csv(f"note-glide_data_earworws.csv") # IMPROV or SONG-FORM